In [7]:
# 1. INSTALL DAN IMPORT LIBRARY
%pip install Sastrawi nltk scikit-learn google-play-scraper pandas

import pandas as pd
import numpy as np
import re
import nltk
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

# download daftar stopword Bahasa Indonesia
nltk.download('stopwords')
from nltk.corpus import stopwords

Note: you may need to restart the kernel to use updated packages.


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\DELL\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [8]:
# 2. LOAD DATASET (HASIL SCRAPING)

# Membaca file hasil scraping Tokopedia
df = pd.read_csv('dataset_dana.csv')

# Menampilkan jumlah baris data
print("Jumlah data:", len(df))

# Menampilkan 5 baris pertama
df.head()


Jumlah data: 10000


,user_name,score,content
0,Irham Alfurqon,5,"mantap PLN, saya bisa lapor keluhan dan respon..."
1,IRFAN EKSEL,5,sangat senang menggunakan aplikasi dana apalag...
2,Kurnia Kur,1,duit gua ilang ka tangung jawab
3,Sakura Rara,5,Kren
4,Roreng Belk,4,lumayan


In [9]:
# 3. CLEANING DASAR

def cleaning_basic(text):
    text = str(text).lower()                 # ubah ke huruf kecil
    text = re.sub(r'http\S+', ' ', text)     # hapus URL
    text = re.sub(r'[^a-z\s]', ' ', text)    # hapus angka dan simbol
    text = re.sub(r'\s+', ' ', text).strip() # hapus spasi berlebih
    return text

# Terapkan ke kolom content
df['clean_content'] = df['content'].apply(cleaning_basic)

# Lihat hasilnya
df[['content', 'clean_content']].head()


,content,clean_content
0,"mantap PLN, saya bisa lapor keluhan dan respon...",mantap pln saya bisa lapor keluhan dan respon ...
1,sangat senang menggunakan aplikasi dana apalag...,sangat senang menggunakan aplikasi dana apalag...
2,duit gua ilang ka tangung jawab,duit gua ilang ka tangung jawab
3,Kren,kren
4,lumayan,lumayan


In [10]:
# 4. NORMALISASI KATA TIDAK BAKU / SLANG

kamus_normalisasi = {
    'gk': 'tidak', 'ga': 'tidak', 'nggak': 'tidak', 'gak': 'tidak',
    'bgt': 'banget', 'bener': 'benar', 'nyesel': 'menyesal',
    'rekomen': 'rekomendasi', 'mantul': 'mantap', 'trima': 'terima',
    'smoga': 'semoga', 'kmu': 'kamu', 'udh': 'sudah', 'sdh': 'sudah',
    'tp': 'tapi', 'jd': 'jadi', 'dgn': 'dengan', 'blm': 'belum',
    'trs': 'terus', 'bkn': 'bukan', 'yg': 'yang', 'brg': 'barang'
}

def normalisasi(text):
    return ' '.join([kamus_normalisasi.get(w, w) for w in text.split()])

df['clean_content'] = df['clean_content'].apply(normalisasi)
df[['content', 'clean_content']].head()


,content,clean_content
0,"mantap PLN, saya bisa lapor keluhan dan respon...",mantap pln saya bisa lapor keluhan dan respon ...
1,sangat senang menggunakan aplikasi dana apalag...,sangat senang menggunakan aplikasi dana apalag...
2,duit gua ilang ka tangung jawab,duit gua ilang ka tangung jawab
3,Kren,kren
4,lumayan,lumayan


In [11]:
df = df = (
    df.groupby('score', group_keys=False)
      .sample(n=400, replace=False, random_state=42)
      .reset_index(drop=True)
)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   user_name      2000 non-null   str  
 1   score          2000 non-null   int64
 2   content        2000 non-null   str  
 3   clean_content  2000 non-null   str  
dtypes: int64(1), str(3)
memory usage: 62.6 KB


In [12]:
# 5. TOKENISASI, STOPWORD REMOVAL, STEMMING

# Inisialisasi stopwords Bahasa Indonesia
stop_words = set(stopwords.words('indonesian'))

# Inisialisasi stemmer dari library Sastrawi
factory = StemmerFactory()
stemmer = factory.create_stemmer()

# Fungsi preprocessing: tokenisasi + hapus stopword + stemming
def preprocess(text):
    tokens = [w for w in text.split() if w not in stop_words]  # tokenisasi & hapus stopword
    stems = [stemmer.stem(w) for w in tokens]                  # stemming
    return ' '.join(stems)

# Terapkan ke kolom teks bersih
df['clean_content'] = df['clean_content'].apply(preprocess)

# Tampilkan contoh hasil
df[['content', 'clean_content']].head()

,content,clean_content
0,eror min gk bisa pkek qriss,eror min pkek qriss
1,aplikasi nya sih bangua cuma tolong lah masala...,aplikasi nya sih bangua tolong dana cicil limi...
2,nga bisa dibuka....uang di dalam ng abisa di a...,nga buka uang ng abisa akses
3,saya membeli satu produk di game tapi saya sud...,beli produk game bayar produk muncul game uang...
4,bagus,bagus


In [13]:
# 6. LEXICON-BASED LABELING (DARI TEKS, BUKAN RATING)

# Kamus kata positif dan negatif (relevan untuk konteks Tokopedia)
positives = {
    "bagus","mantap","puas","rekomendasi","cepat","murah","nyaman","baik",
    "keren","berfungsi","ori","original","sesuai","rapi","worth","memuaskan",
    "mantapp","top","oke","responsif","ramah","tepat","aman","senang","mantul"
}

negatives = {
    "buruk","kecewa","lemot","lambat","error","parah","jelek","mengecewakan",
    "rusak","telat","bohong","tipu","refund","lama","hang","crash","bug",
    "down","delay","ribet","susah","gagal","macet","lemotnya","tidakfungsi"
}

# Kata penguat (intensifier) dan penyangkal (negator)
intensifier = {"sangat","banget","amat","terlalu"}
negator = {"tidak","tak","nggak","ga","gak","bukan"}

# Fungsi perhitungan skor lexicon
def lexicon_score(tokens_str):
    score, prev = 0, ""
    for t in tokens_str.split():
        s = 1 if t in positives else (-1 if t in negatives else 0)
        # negasi → membalik skor
        if prev in negator and s != 0:
            s = -s
        # penguat → menggandakan skor
        if prev in intensifier and s != 0:
            s *= 2
        score += s
        prev = t
    return score

# Fungsi untuk menentukan label berdasarkan skor total
def label_from_text(tokens_str):
    s = lexicon_score(tokens_str)
    if s > 0:
        return "positif"
    elif s < 0:
        return "negatif"
    else:
        return "netral"

# Terapkan ke seluruh data
df['sentimen'] = df['clean_content'].apply(label_from_text)

# Lihat distribusi hasil label
print(df['sentimen'].value_counts())
df[['content','clean_content','sentimen']].head()

sentimen
netral     1288
positif     490
negatif     222
Name: count, dtype: int64


,content,clean_content,sentimen
0,eror min gk bisa pkek qriss,eror min pkek qriss,netral
1,aplikasi nya sih bangua cuma tolong lah masala...,aplikasi nya sih bangua tolong dana cicil limi...,positif
2,nga bisa dibuka....uang di dalam ng abisa di a...,nga buka uang ng abisa akses,netral
3,saya membeli satu produk di game tapi saya sud...,beli produk game bayar produk muncul game uang...,netral
4,bagus,bagus,positif


In [14]:
# 7. SPLIT DATA (SETELAH PREPROCESSING LENGKAP)

from sklearn.model_selection import train_test_split

# X = teks bersih; y = label hasil analisis teks (bukan rating!)
X = df['clean_content']
y = df['sentimen']

# Bagi data 80% untuk training, 20% untuk testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Tampilkan jumlah masing-masing
print("Data latih :", len(X_train))
print("Data uji   :", len(X_test))


Data latih : 1600
Data uji   : 400


In [15]:
# 8. EKSTRAKSI FITUR (TF-IDF)

from sklearn.feature_extraction.text import TfidfVectorizer

# Membuat objek TF-IDF dengan maksimal 5000 fitur
vectorizer = TfidfVectorizer(max_features=5000)

# Latih TF-IDF di data training, lalu transformasi data training & testing
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf  = vectorizer.transform(X_test)

# Tampilkan bentuk matriks (baris = jumlah data, kolom = jumlah fitur)
print("Shape TF-IDF:")
print("Train:", X_train_tfidf.shape)
print("Test :", X_test_tfidf.shape)

Shape TF-IDF:
Train: (1600, 1952)
Test : (400, 1952)


In [16]:
# 9. TRAINING 3 MODEL DAN EVALUASI

from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

# Siapkan model
models = {
    "Logistic Regression": LogisticRegression(max_iter=300),
    "SVM": LinearSVC(),
    "Random Forest": RandomForestClassifier(n_estimators=150, random_state=42)
}

results = {}

# Latih dan uji tiap model
for name, m in models.items():
    m.fit(X_train_tfidf, y_train)                   # Latih model
    pred = m.predict(X_test_tfidf)                  # Prediksi di data uji
    acc = accuracy_score(y_test, pred)              # Hitung akurasi
    results[name] = acc
    print(f"\n=== {name} ===")
    print(f"Akurasi: {acc*100:.2f}%")
    print(classification_report(y_test, pred))

# Pilih model terbaik
best_model_name = max(results, key=results.get)
best_model = models[best_model_name]
print(f"\nModel terbaik: {best_model_name} (akurasi {results[best_model_name]*100:.2f}%)")


=== Logistic Regression ===
Akurasi: 88.50%
              precision    recall  f1-score   support

     negatif       0.95      0.43      0.59        44
      netral       0.87      0.97      0.92       258
     positif       0.92      0.86      0.89        98

    accuracy                           0.89       400
   macro avg       0.91      0.75      0.80       400
weighted avg       0.89      0.89      0.88       400


=== SVM ===
Akurasi: 92.25%
              precision    recall  f1-score   support

     negatif       0.96      0.59      0.73        44
      netral       0.92      0.97      0.94       258
     positif       0.92      0.95      0.93        98

    accuracy                           0.92       400
   macro avg       0.93      0.84      0.87       400
weighted avg       0.92      0.92      0.92       400


=== Random Forest ===
Akurasi: 92.50%
              precision    recall  f1-score   support

     negatif       0.96      0.61      0.75        44
      netral    

In [17]:
# 10. INFERENCE (UJI KALIMAT BARU)

def prediksi_sentimen(teks):
    # Gunakan pipeline preprocessing yang sama seperti sebelumnya
    teks = cleaning_basic(teks)
    teks = normalisasi(teks)
    teks = preprocess(teks)
    X_new = vectorizer.transform([teks])
    return best_model.predict(X_new)[0]

# Uji beberapa contoh kalimat baru
contoh = [
    "Aplikasinya bagus banget dan pengiriman cepat!",
    "Biasa saja, kadang error pas transaksi.",
    "Kecewa banget, aplikasi sering force close."
]

for c in contoh:
    print(f"{c} -> {prediksi_sentimen(c)}")

Aplikasinya bagus banget dan pengiriman cepat! -> positif
Biasa saja, kadang error pas transaksi. -> negatif
Kecewa banget, aplikasi sering force close. -> negatif


In [18]:
# 11. SIMPAN DATASET BERSIH DAN FILE REQUIREMENTS

# Simpan dataset akhir
df.to_csv('clean_dataset_final.csv', index=False, encoding='utf-8')
print("File 'clean_dataset_final.csv' berhasil disimpan.")

# Buat daftar library yang dipakai
!pip freeze > requirements.txt
print("File 'requirements.txt' berhasil dibuat.")


File 'clean_dataset_final.csv' berhasil disimpan.
File 'requirements.txt' berhasil dibuat.
